# Mining Concession Land-Change Detection — RasterFlow Pipeline

**Why a mining operator runs this.** ESG disclosure, TCFD / TNFD
reporting, and lender covenants increasingly require the operator to
quantify direct and indirect land impact of each active concession,
quarterly, with spatial evidence. Ground surveys don't scale; annual
Landsat analytics are too coarse. RasterFlow plus Sentinel-2 L2A
closes the gap: 10 m resolution, pan-tropical coverage, automated.

This notebook is the end-to-end pipeline:

1. **AOI** — a sub-polygon of the Carajás iron-ore complex in Pará,
   Brazil — one of the world's largest mining districts surrounded by
   primary Amazonian rainforest.
2. **Bi-temporal mosaic** — `rf_client.build_mosaic` fetches two years of
   Sentinel-2 and composites a dense-season mosaic per time step.
3. **Change detection** — `rf_client.predict_mosaic` runs a segmentation
   model with the `SEMANTIC_SEGMENTATION_CHANGE_DETECTION_PYTORCH`
   actor; the two time steps are stacked along the channel dimension
   so a single forward pass yields per-pixel change labels.
4. **Vectorize** — `rf_client.vectorize_mosaic` converts the change
   raster into polygon geometries with confidence scores.
5. **Quantify** — project to UTM 22S, sum hectares by disturbance type
   and write a managed Iceberg table the sustainability team's tooling
   reads directly.

> **Demo scope.** §3-§5 show the production RasterFlow API pattern —
> the same calls validated in `Analyzing_Data/RasterFlow_ChangeDetection.ipynb`.
> Running the mosaic + inference end-to-end is ~40 min wall time, so
> §6 constructs twelve realistic disturbance polygons inside the AOI
> (forest clearing, haul roads, pads, tailings) and §7-§10 run the
> hectare math, classification, and persistence in seconds. Swap the
> synthetic polygons for the vectorized change output and the rest of
> the pipeline is unchanged.

## 1. Setup

In [ ]:
from sedona.spark import *
import pyspark.sql.functions as f
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType
)
import json

config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)

# UTM zone 22S covers Pará / Carajás — equal-area enough at local scale
# for hectare accounting (sub-0.5% error within the concession footprint).
AREA_CRS = "EPSG:32722"

TARGET_DATABASE = "land_change_demo"
TARGET_TABLE    = "carajas_disturbance"
TARGET_FQN      = f"org_catalog.{TARGET_DATABASE}.{TARGET_TABLE}"

## 2. AOI — Carajás Sub-Concession

A ~12,200 ha sub-polygon of the Carajás complex. In production this is
the operator's official concession shape loaded from the GIS system.

In [ ]:
AOI_BBOX = (-50.15, -6.15, -50.05, -6.05)   # (min_lon, min_lat, max_lon, max_lat)

aoi_df = sedona.sql(f"""
    SELECT
        'Carajás Sub-Concession A' AS concession_id,
        ST_MakeEnvelope(
            {AOI_BBOX[0]}, {AOI_BBOX[1]},
            {AOI_BBOX[2]}, {AOI_BBOX[3]}, 4326
        ) AS geometry
""")
aoi_df.createOrReplaceTempView("concession_aoi")

aoi_ha = sedona.sql(f"""
    SELECT ROUND(
        ST_Area(ST_Transform(geometry, 'EPSG:4326', '{AREA_CRS}')) / 10000.0, 0
    ) AS aoi_hectares
    FROM concession_aoi
""").first().aoi_hectares
print(f"AOI: Carajás Sub-Concession A — {aoi_ha:,.0f} hectares")

## 3. Stage 1 — Build the Bi-Temporal Mosaic

`rf_client.build_mosaic` with `DatasetEnum.S2_MED_WINDOWED_PIXEL` over a
two-year window produces a Zarr store with two time steps — one per
year, cloud-filtered and pixel-median-composited so each pixel
represents the stable dense-season observation.

```python
from rasterflow_remote import RasterflowClient, DatasetEnum, InferenceActorEnum
from rasterflow_remote.data_models import MergeModeEnum, VectorizeMethodEnum
from datetime import datetime

rf = RasterflowClient()

mosaic_index = rf.build_mosaic(
    datasets = [DatasetEnum.S2_MED_WINDOWED_PIXEL],
    aoi      = aoi_path,                 # GeoParquet on S3
    start    = datetime(2022, 6, 1),
    end      = datetime(2024, 6, 1),
)
input_store = mosaic_index.first_row_mosaic
```

Wall time: ~5 minutes for this AOI.

## 4. Stage 2 — Change Detection Inference

The change-detection actor stacks the two time steps along the channel
dimension (so each sample is `(Bands × 2, H, W)`) and feeds the
8-channel tensor through a segmentation model that outputs per-pixel
change labels. For the canonical canopy-loss story you'd use a
forest-loss model; the RasterFlow_ChangeDetection reference notebook
uses FTW, and the same call shape applies here.

```python
model_output_index = rf.predict_mosaic(
    store       = input_store,
    model_path  = "s3://<operator-bucket>/models/forest_change-v1.pt2",
    patch_size  = 256,
    clip_size   = 32,
    device      = "cuda",
    features    = [
        "s2med_windowed_pixel:B04",  # red,   time 1
        "s2med_windowed_pixel:B03",  # green, time 1
        "s2med_windowed_pixel:B02",  # blue,  time 1
        "s2med_windowed_pixel:B08",  # NIR,   time 1
    ],
    labels      = ['stable', 'deforested', 'infrastructure'],
    actor       = InferenceActorEnum.SEMANTIC_SEGMENTATION_CHANGE_DETECTION_PYTORCH,
    max_batch_size     = 128,
    merge_mode         = MergeModeEnum.WEIGHTED_AVERAGE,
    xy_block_multiplier= 1,
)
model_output_store = model_output_index.first_row_mosaic
```

Wall time: ~30 minutes for this AOI on a GPU runtime.

## 5. Stage 3 — Vectorize to Polygons

Threshold the change raster and trace polygon boundaries. Output is a
GeoParquet with `geometry` (the polygon), `label` (the change class),
and `score_mean` (confidence).

```python
change_vectors = rf.vectorize_mosaic(
    store            = model_output_store,
    features         = ['deforested', 'infrastructure'],
    threshold        = 0.5,
    vectorize_method = VectorizeMethodEnum.SEMANTIC_SEGMENTATION_RASTERIO,
    vectorize_config = {'stats': True},
)
disturbance_df = (
    sedona.read.format('geoparquet').load(change_vectors.uri)
          .withColumn('geometry', f.expr('ST_MakeValid(geometry)'))
          .withColumn('geometry', f.expr('ST_SetSRID(geometry, 4326)'))
          .withColumn('geometry',
                      f.expr('ST_SimplifyPreserveTopology(geometry, 0.00002)'))
)
```

## 6. Demo Disturbance Polygons — Runnable from Here

Twelve polygons inside the concession representing the kinds of change
the pipeline produces: three large clearings next to active pits, four
linear haul-road/conveyor extensions, three infrastructure pads, and two
tailings-expansion zones. Swap these for the real vectorized change
polygons in production; the rest of the pipeline is identical.

In [ ]:
# (id, label, class, confidence, min_lon, min_lat, max_lon, max_lat, observed_year)
disturbance_rows = [
    # Large forest clearings
    ("CHG-001", "North pit expansion",      "deforested",     0.94,
        -50.140, -6.075, -50.120, -6.062, 2024),
    ("CHG-002", "SW flank clearing",        "deforested",     0.91,
        -50.135, -6.125, -50.118, -6.112, 2024),
    ("CHG-003", "East buffer encroachment", "deforested",     0.87,
        -50.078, -6.090, -50.065, -6.080, 2024),
    # Haul roads / conveyor corridors (long, narrow)
    ("CHG-004", "Haul road extension A",    "infrastructure", 0.92,
        -50.128, -6.108, -50.084, -6.105, 2024),
    ("CHG-005", "Haul road extension B",    "infrastructure", 0.89,
        -50.112, -6.140, -50.110, -6.090, 2024),
    ("CHG-006", "Conveyor corridor N-S",    "infrastructure", 0.85,
        -50.100, -6.145, -50.098, -6.080, 2023),
    ("CHG-007", "Access spur east",         "infrastructure", 0.83,
        -50.078, -6.128, -50.059, -6.127, 2024),
    # Infrastructure pads
    ("CHG-008", "Workshop pad",             "infrastructure", 0.88,
        -50.105, -6.083, -50.100, -6.080, 2023),
    ("CHG-009", "Fuel depot pad",           "infrastructure", 0.86,
        -50.091, -6.097, -50.088, -6.094, 2024),
    ("CHG-010", "Explosives magazine",      "infrastructure", 0.82,
        -50.082, -6.113, -50.080, -6.111, 2024),
    # Tailings expansions
    ("CHG-011", "Tailings cell 3 exp.",     "tailings",       0.93,
        -50.148, -6.108, -50.130, -6.092, 2024),
    ("CHG-012", "Tailings cell 4 berm",     "tailings",       0.90,
        -50.128, -6.080, -50.118, -6.073, 2024),
]

schema = StructType([
    StructField("change_id",     StringType()),
    StructField("label",         StringType()),
    StructField("class",         StringType()),
    StructField("confidence",    DoubleType()),
    StructField("min_lon",       DoubleType()),
    StructField("min_lat",       DoubleType()),
    StructField("max_lon",       DoubleType()),
    StructField("max_lat",       DoubleType()),
    StructField("observed_year", IntegerType()),
])

disturb_df = sedona.createDataFrame(disturbance_rows, schema) \
    .withColumn(
        "geometry",
        f.expr("ST_MakeEnvelope(min_lon, min_lat, max_lon, max_lat, 4326)")
    )
disturb_df.createOrReplaceTempView("disturbance_polygons")

print(f"Disturbance polygons: {disturb_df.count()}")
disturb_df.select("change_id", "label", "class", "confidence",
                  "observed_year").show(truncate=False)

## 7. Stage 4 — Quantify in Hectares

Project each polygon to UTM 22S (`EPSG:32722`), compute area in m²,
convert to hectares. One SQL — this is the deliverable math.

In [ ]:
quantified_df = sedona.sql(f"""
    SELECT
        change_id, label, class, confidence, observed_year,
        geometry,
        ROUND(
            ST_Area(ST_Transform(geometry, 'EPSG:4326', '{AREA_CRS}'))
            / 10000.0, 2
        ) AS area_ha,
        CASE
            WHEN class = 'deforested'     THEN 'DEFORESTATION'
            WHEN class = 'infrastructure' THEN 'LAND_DISTURBANCE'
            WHEN class = 'tailings'       THEN 'TAILINGS_EXPANSION'
            ELSE                                'OTHER'
        END AS disclosure_category
    FROM disturbance_polygons
""").cache()
quantified_df.createOrReplaceTempView("disturbance_quantified")

quantified_df.orderBy(f.col("area_ha").desc()) \
             .select("change_id", "label", "disclosure_category",
                     "area_ha", "observed_year", "confidence") \
             .show(truncate=False)

## 8. Disclosure Rollup — Hectares by Category and Year

In [ ]:
sedona.sql("""
    SELECT
        disclosure_category,
        COUNT(*)                            AS polygon_count,
        ROUND(SUM(area_ha), 2)              AS total_ha,
        ROUND(AVG(area_ha), 2)              AS avg_ha,
        ROUND(MAX(area_ha), 2)              AS largest_single_ha,
        ROUND(AVG(confidence), 3)           AS avg_confidence
    FROM disturbance_quantified
    GROUP BY disclosure_category
    ORDER BY total_ha DESC
""").show(truncate=False)

sedona.sql("""
    SELECT
        observed_year,
        disclosure_category,
        ROUND(SUM(area_ha), 2) AS total_ha
    FROM disturbance_quantified
    GROUP BY observed_year, disclosure_category
    ORDER BY observed_year, disclosure_category
""").show(truncate=False)

# Concession-wide headline
headline = sedona.sql(f"""
    SELECT
        COUNT(*)                                      AS total_changes,
        ROUND(SUM(area_ha), 1)                        AS total_disturbed_ha,
        ROUND(SUM(CASE WHEN disclosure_category = 'DEFORESTATION'
                       THEN area_ha ELSE 0 END), 1)   AS deforestation_ha,
        ROUND(SUM(area_ha) / {aoi_ha} * 100.0, 2)     AS pct_of_concession
    FROM disturbance_quantified
""").first()

print("\n══════════════════════════════════════════════")
print(f"  {headline.total_changes:>6}  change polygons detected")
print(f"  {headline.total_disturbed_ha:>6.1f}  hectares total land disturbance")
print(f"  {headline.deforestation_ha:>6.1f}  hectares deforestation")
print(f"  {headline.pct_of_concession:>6.2f}%  of concession area")
print("══════════════════════════════════════════════")

## 9. Persist to Iceberg

One row per change polygon with geometry + area + category. The
sustainability reporting stack reads directly from here.

In [ ]:
sedona.sql(f"CREATE DATABASE IF NOT EXISTS org_catalog.{TARGET_DATABASE}")

sedona.sql(f"""
    CREATE OR REPLACE TABLE {TARGET_FQN} AS
    SELECT
        change_id, label, class, disclosure_category,
        confidence, observed_year, area_ha,
        geometry
    FROM disturbance_quantified
""")
print(f"Wrote {TARGET_FQN}")
sedona.sql(f"SELECT COUNT(*) AS rows, ROUND(SUM(area_ha), 1) AS total_ha "
           f"FROM {TARGET_FQN}").show()

## 10. Export GeoJSON for the Sustainability Dashboard

Two FeatureCollections — the concession boundary and the change
polygons — ready for a map view in the reporting portal.

In [ ]:
def df_to_fc(df, geom_col, prop_cols):
    rows = df.selectExpr(*prop_cols, f"ST_AsGeoJSON({geom_col}) AS gj").collect()
    feats = []
    for r in rows:
        d = r.asDict()
        gj = d.pop("gj")
        feats.append({
            "type": "Feature",
            "properties": {k: (float(v) if isinstance(v, (int, float)) else v)
                           for k, v in d.items()},
            "geometry": json.loads(gj),
        })
    return {"type": "FeatureCollection", "features": feats}

aoi_fc = df_to_fc(aoi_df, "geometry", ["concession_id"])
with open("/tmp/carajas_concession.geojson", "w") as fh:
    json.dump(aoi_fc, fh)

changes_fc = df_to_fc(
    sedona.sql(f"SELECT * FROM {TARGET_FQN}"),
    "geometry",
    ["change_id", "label", "class", "disclosure_category",
     "confidence", "observed_year", "area_ha"]
)
with open("/tmp/carajas_disturbance.geojson", "w") as fh:
    json.dump(changes_fc, fh)

print("Wrote /tmp/carajas_concession.geojson "
      f"({len(aoi_fc['features'])} feature)")
print("Wrote /tmp/carajas_disturbance.geojson "
      f"({len(changes_fc['features'])} features)")

## 11. Preview on a Map

In [ ]:
viz = SedonaKepler.create_map()
SedonaKepler.add_df(viz,
    aoi_df.select("concession_id", "geometry"),
    name="Concession Boundary")
SedonaKepler.add_df(viz,
    quantified_df.select("change_id", "label", "disclosure_category",
                         "area_ha", "observed_year", "geometry"),
    name="Disturbance Polygons")
viz

---

## From demo to production

| Stage | Demo | Production |
|---|---|---|
| AOI | Bounding envelope | Operator's official concession polygon from GIS |
| Mosaic | (call pattern shown) | `rf.build_mosaic(S2_MED_WINDOWED_PIXEL, 2-yr window)` |
| Inference | (call pattern shown) | `rf.predict_mosaic(..., SEMANTIC_SEGMENTATION_CHANGE_DETECTION_PYTORCH)` with a trained forest-loss or land-cover-change model |
| Disturbance polygons | 12 synthesized features | `rf.vectorize_mosaic(..., threshold=0.5)` output |
| Area | UTM 22S (same) | Same — pick the UTM zone covering the concession |
| Sink | `org_catalog.land_change_demo.*` | Operator's managed catalog; scheduled quarterly refresh |

## Outputs

| Artifact | Contents |
|---|---|
| `org_catalog.land_change_demo.carajas_disturbance` | One Iceberg row per change polygon with geometry, area_ha, disclosure_category |
| `/tmp/carajas_concession.geojson` | Concession boundary for basemap |
| `/tmp/carajas_disturbance.geojson` | Change polygons colored by disclosure category |